In [11]:
# Naive Bayes Classifier

import pandas as pd

def calculate_prior_probabilities(y):
    return y.value_counts(normalize=True)

def naive_bayes_classifier(X_test, priors, likelihoods):
    predictions = []
    for _, data_point in X_test.iterrows():
        class_probabilities = {}
        for class_ in priors.index:
            class_probabilities[class_] = priors[class_]
            for feature in X_test.columns:
                val = data_point[feature]
                # Get smoothed probability
                prob = likelihoods[feature][class_].get(val, 1e-6)
                class_probabilities[class_] *= prob

        predictions.append(max(class_probabilities, key=class_probabilities.get))

    return predictions

def calculate_likelihoods_with_smoothing(X, y):
    likelihoods = {}
    for column in X.columns:
        likelihoods[column] = {}
        unique_feature_values = X[column].unique()
        k = len(unique_feature_values)  # Number of unique categories in feature
        
        for class_ in y.unique():
            class_data = X[y == class_][column]
            total_count = len(class_data)
            feature_counts = class_data.value_counts()
            
            # Calculate Laplace smoothing for every unique value
            probs = {}
            for val in unique_feature_values:
                count = feature_counts.get(val, 0)
                probs[val] = (count + 1) / (total_count + k)
                
            likelihoods[column][class_] = probs
            
    return likelihoods

# --- Dataset ---
data = {
    'Temperature': ['Hot', 'Hot', 'Cold', 'Hot', 'Cold', 'Cold', 'Cold'],
    'Humidity': ['High', 'High', 'Normal', 'Normal', 'High', 'Normal', 'Normal'],
    'Weather': ['Sunny', 'Sunny', 'Snowy', 'Rainy', 'Snowy', 'Snowy', 'Sunny']
}
df = pd.DataFrame(data)

# Split features and labels
X = df[['Temperature', 'Humidity']]
y = df['Weather']

# Calculate prior probabilities
priors = calculate_prior_probabilities(y)

# Calculate likelihoods with smoothing
likelihoods = calculate_likelihoods_with_smoothing(X, y)

# New observation
X_test = pd.DataFrame([{'Temperature': 'Cold', 'Humidity': 'Normal'}])

# Make prediction
prediction = naive_bayes_classifier(X_test, priors, likelihoods)
print("Predicted Weather:", prediction[0])

Predicted Weather: Snowy
